[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qussai96/ProtAudit/blob/main/ProtAudit_Colab.ipynb)

# ProtAudit: score protein sequences

Paste **1–100 protein sequences in FASTA format** and run the frozen ProtAudit ProtT5 model. Higher scores indicate a more protein-like sequence.

The notebook produces:

- a TSV file with one protein-likeness score per FASTA record;
- the frozen validation-threshold call for each protein;
- a score-band summary plot;
- a score-distribution histogram; and
- a ZIP archive containing the table and both plots.

**Runtime:** choose **Runtime → Change runtime type → GPU** before running. ProtT5 model weights are downloaded from Hugging Face during the first run, so setup can take several minutes.

In [ ]:
#@title 1. Install and load ProtAudit
import json
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError("This notebook is designed to run in Google Colab.") from exc

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.38,<5", "sentencepiece>=0.2", "matplotlib>=3.7"
], check=True)

REPOSITORY_URL = "https://github.com/qussai96/ProtAudit.git"
REPOSITORY_DIR = Path("/content/ProtAudit")
if not REPOSITORY_DIR.exists():
    subprocess.run([
        "git", "clone", "--quiet", "--depth", "1",
        REPOSITORY_URL, str(REPOSITORY_DIR)
    ], check=True)
elif (REPOSITORY_DIR / ".git").is_dir():
    subprocess.run([
        "git", "-C", str(REPOSITORY_DIR), "pull", "--quiet", "--ff-only"
    ], check=True)

required = [
    REPOSITORY_DIR / "embed.py",
    REPOSITORY_DIR / "score.py",
    REPOSITORY_DIR / "models/manifest.json",
    REPOSITORY_DIR / "models/prott5.pt",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Incomplete ProtAudit checkout: " + ", ".join(missing))

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Choose Runtime → Change runtime type → GPU, "
        "then rerun the notebook from the beginning."
    )

manifest = json.loads((REPOSITORY_DIR / "models/manifest.json").read_text())
model_info = manifest["models"]["prott5"]
print("GPU:", torch.cuda.get_device_name(0))
print("ProtAudit repository loaded.")
print(f"Frozen ProtT5 threshold: {model_info['validation_threshold']:.8f}")

## Paste protein sequences

Requirements:

- paste sequences in FASTA format, including one header beginning with `>` per protein;
- identifiers (the first token after `>`) must be unique;
- sequences may contain the 20 standard amino acids or `X`; and
- the file must contain no more than 100 proteins.

Long proteins are processed using non-overlapping 1,000-residue windows and a residue-weighted mean; residues are not truncated.

In [ ]:
#@title 2. Paste protein FASTA (maximum 100 proteins)
import ipywidgets as widgets
from IPython.display import clear_output, display

MAX_SEQUENCES = 100
INPUT_FASTA = Path("/content/protaudit_input.faa")
sys.path.insert(0, str(REPOSITORY_DIR))
from embed import read_fasta

fasta_box = widgets.Textarea(
    value="",
    placeholder=">protein_1\nMKT...\n>protein_2\nMAV...",
    description="FASTA:",
    layout=widgets.Layout(width="100%", height="260px"),
    style={"description_width": "60px"},
)
accept_button = widgets.Button(
    description="Use pasted FASTA", button_style="success", icon="check"
)
status_output = widgets.Output()

def accept_pasted_fasta(_):
    global records
    with status_output:
        clear_output()
        fasta_text = fasta_box.value.strip()
        if not fasta_text:
            records = []
            print("Please paste at least one FASTA record.")
            return
        try:
            INPUT_FASTA.write_text(fasta_text + "\n")
            candidate_records = list(read_fasta(INPUT_FASTA))
            if not candidate_records:
                raise ValueError("No protein records were found.")
            if len(candidate_records) > MAX_SEQUENCES:
                raise ValueError(
                    f"Found {len(candidate_records)} proteins; this Colab is limited "
                    f"to {MAX_SEQUENCES}. Use the local ProtAudit release for larger inputs: "
                    f"{REPOSITORY_URL}"
                )
            records = candidate_records
            lengths = [len(sequence) for _, sequence in records]
            print(f"Accepted {len(records)} protein(s). You may now run step 3.")
            print(f"Length range: {min(lengths):,}–{max(lengths):,} aa")
            print(f"Total residues: {sum(lengths):,}")
        except Exception as exc:
            records = []
            print(f"Invalid FASTA: {exc}")

accept_button.on_click(accept_pasted_fasta)
display(widgets.HTML("<b>Paste FASTA records below, then click Use pasted FASTA.</b>"))
display(fasta_box, accept_button, status_output)

## Generate ProtT5 embeddings and score proteins

The embedding step is the slow part. The code automatically splits a batch if the GPU runs out of memory.

In [ ]:
#@title 3. Generate ProtT5 embeddings
EMBEDDING_DIR = Path("/content/protaudit_embeddings")

command = [
    sys.executable, str(REPOSITORY_DIR / "embed.py"), str(INPUT_FASTA),
    "--output", str(EMBEDDING_DIR),
    "--model", "prott5",
    "--device", "cuda",
    "--max-batch-tokens", "4000",
    "--max-batch-size", "8",
    "--overwrite",
]
subprocess.run(command, check=True)

embedding_summary = json.loads((EMBEDDING_DIR / "summary.json").read_text())
if embedding_summary.get("record_count") != len(records):
    raise RuntimeError("Embedding row count does not match the pasted FASTA.")
print("Embedding validation passed:", embedding_summary["embedding_shape"])

In [ ]:
#@title 4. Score, display, and plot the proteins
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

OUTPUT_DIR = Path("/content/ProtAudit_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCORE_TABLE = OUTPUT_DIR / "protaudit_scores.tsv"
SCORE_PLOT = OUTPUT_DIR / "protaudit_scores_plot.png"
HISTOGRAM_PLOT = OUTPUT_DIR / "protaudit_score_distribution.png"

subprocess.run([
    sys.executable, str(REPOSITORY_DIR / "score.py"), str(EMBEDDING_DIR),
    "--output", str(SCORE_TABLE),
    "--plot", str(SCORE_PLOT),
], check=True)

results = pd.read_csv(SCORE_TABLE, sep="\t")
if len(results) != len(records) or results["protein_id"].tolist() != [x[0] for x in records]:
    raise RuntimeError("Score-table identifiers do not match the pasted FASTA.")

scores = results["protein_likeness_score"].astype(float)
bin_count = min(20, max(5, int(np.ceil(np.sqrt(len(scores))))))
fig, ax = plt.subplots(figsize=(7.2, 4.5))
ax.hist(scores, bins=np.linspace(0, 1, bin_count + 1), color="#3C7DA6", edgecolor="white", linewidth=0.8)
ax.axvline(model_info["validation_threshold"], color="#D54E53", linestyle="--", linewidth=1.8,
           label=f"Frozen threshold = {model_info['validation_threshold']:.3f}")
ax.axvline(scores.median(), color="#333333", linestyle=":", linewidth=1.8,
           label=f"Median = {scores.median():.3f}")
ax.set(xlim=(0, 1), xlabel="ProtAudit protein-likeness score", ylabel="Number of proteins")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig(HISTOGRAM_PLOT, dpi=300, bbox_inches="tight")
plt.close(fig)

display(results)
display(Image(filename=str(SCORE_PLOT)))
display(Image(filename=str(HISTOGRAM_PLOT)))

In [ ]:
#@title 5. Download the scores and plots
import shutil

archive = Path(shutil.make_archive(
    "/content/ProtAudit_results", "zip", root_dir=OUTPUT_DIR
))
print(f"Downloading {archive.name} ({archive.stat().st_size / 1024:.1f} KiB)")
files.download(str(archive))

## Interpretation

- `protein_likeness_score` ranges from 0 to 1; higher scores are more protein-like.
- `passes_frozen_threshold` uses the ProtT5 threshold selected by maximum F1 on the frozen plant validation species (**0.45833838**). It was not fitted to the pasted FASTA.
- `score_band` divides scores into `>=0.9`, `0.5–0.9`, and `<0.5` descriptive bands. These bands are not calibrated probabilities or separate trained thresholds.
- A low score prioritizes a model for inspection; it does not by itself identify the underlying genomic annotation error.
- For files larger than 100 proteins, use the [local ProtAudit release](https://github.com/qussai96/ProtAudit).